# Jinx! Learning Simulation


## Install and Import Dependencies


In [ ]:
import numpy as np
from functools import reduce
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import string
import os
from tqdm import tqdm

# Set Computer Modern font for all plots
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Computer Modern Roman', 'CMU Serif', 'DejaVu Serif', 'Times New Roman']
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['axes.unicode_minus'] = False


## Define All Classes and Functions


In [301]:
class Game:
    def __init__(self, payoffs):
        self.payoffs = payoffs


class Learner:
    def __init__(self, game, role):
        self.game = game
        self.role = role
        self.num_actions = game.payoffs[0].shape[role]
        self.current_strategy = None

    def observe_utility(self, utility):
        pass

    def next_strategy(self):
        pass


class MWULearner(Learner):
    def __init__(self, game, role, random_steps=0, eta=1):
        super().__init__(game, role)
        self.cum_utilities = np.zeros(self.num_actions)
        self.steps = 0
        self.eta = eta
        self.random_steps = random_steps

    def observe_utility(self, utility):
        self.cum_utilities += utility
        self.cum_utilities -= self.cum_utilities.max()
        self.steps += 1

    def next_strategy(self):
        # Random Steps
        if self.steps < self.random_steps:
            random_strategy = np.random.random(self.num_actions)
            random_strategy /= random_strategy.sum()
            self.current_strategy = random_strategy
            return random_strategy

        # MWU
        strategy = np.exp(self.cum_utilities * self.eta)
        strategy /= strategy.sum()
        self.current_strategy = strategy
        return strategy


class RMLearner(Learner):
    def __init__(self, game, role, random_steps=0):
        super().__init__(game, role)
        self.cum_regret = np.zeros(self.num_actions)
        self.steps = 0
        self.random_steps = random_steps

    def observe_utility(self, utility):
        cur_utility = self.current_strategy @ utility
        cur_regret = utility - cur_utility
        self.cum_regret += cur_regret
        self.steps += 1

    def next_strategy(self):
        # Random Steps
        if self.steps < self.random_steps:
            random_strategy = np.random.random(self.num_actions)
            random_strategy /= random_strategy.sum()
            self.current_strategy = random_strategy
            return random_strategy

        # Regret Matching
        cum_reg_p = np.maximum(self.cum_regret, 0)
        total = cum_reg_p.sum()

        if total == 0:
            strategy = np.ones(self.num_actions) / self.num_actions
        else:
            strategy = cum_reg_p / total

        self.current_strategy = strategy
        return strategy


class LearningAlg:
    def __init__(self, game: Game, learner_types: list, random_steps: int):
        self.game = game
        self.num_learners = len(learner_types)
        assert self.num_learners == game.payoffs[0].ndim, "Incorrect number of learners"
        self.history = []
        self.learners = [learner_type(game, role, random_steps) for role, learner_type in enumerate(learner_types)]
        self.rounds = 0

    def train(self, rounds: int):
        for _ in range(rounds):
            # Next Strategies
            strategies = [learner.next_strategy() for learner in self.learners]
            self.history.append(strategies)

            # Calculate Utilities (multi-player safe)
            utilities = []
            n = self.num_learners
            axes = string.ascii_lowercase

            for i in range(n):
                payoff_i = self.game.payoffs[i]
                idx = axes[:n]
                terms = [idx] + [idx[j] for j in range(n) if j != i]
                equation = ','.join(terms) + '->' + idx[i]
                args = [payoff_i] + [strategies[j] for j in range(n) if j != i]
                utility_i = np.einsum(equation, *args)
                utilities.append(utility_i)

            # Propagate Utilities
            for role, learner in enumerate(self.learners):
                learner.observe_utility(utilities[role])

        self.rounds += rounds

    def average_plays(self):
        avg_plays = np.zeros((self.rounds,) + self.game.payoffs[0].shape)
        total_play = np.zeros(self.game.payoffs[0].shape, dtype=np.float64)

        for round, strategies in enumerate(self.history):
            play = reduce(np.multiply.outer, strategies)
            total_play += play
            avg_plays[round] = total_play / (round + 1)

        return avg_plays

    def average_strategies(self):
        avg_strategies = [np.zeros((self.rounds, learner.num_actions)) for learner in self.learners]
        ttl_strategy = [np.zeros(learner.num_actions) for learner in self.learners]

        for round, strategies in enumerate(self.history):
            for role, strategy in enumerate(strategies):
                ttl_strategy[role] += strategy
                avg_strategies[role][round] = ttl_strategy[role] / (round + 1)

        return avg_strategies

    def get_strategy_history(self):
        strategy_history = [np.zeros((self.rounds, learner.num_actions)) for learner in self.learners]

        for round, strategies in enumerate(self.history):
            for role, strategy in enumerate(strategies):
                strategy_history[role][round] = strategy

        return strategy_history

    def get_play_history(self):
        play_history = np.zeros((self.rounds,) + self.game.payoffs[0].shape)

        for round, strategies in enumerate(self.history):
            play = reduce(np.multiply.outer, strategies)
            play_history[round] = play

        return play_history


def learning(payoffs, T, learner_types, random_steps=0):
    if not isinstance(learner_types, list):
        learner_types = [learner_types] * (payoffs.ndim - 1)
    game = Game(payoffs)
    learning_alg = LearningAlg(game, learner_types, random_steps)
    learning_alg.train(T)
    return learning_alg


def get_jinx_game(n):
    payoffs = np.eye(n)
    payoffs = np.stack([payoffs, payoffs], axis=0)
    game = Game(payoffs)
    return game


def visualize_plays_static(plays, title_prefix="", title_suffix="", save_file=True):
    """Plots n×n many 2D graphs showing the probability of playing each action combination over time."""
    plays_array = np.array(plays)

    if plays_array.ndim == 3:
        plays_list = [plays_array]
    elif plays_array.ndim == 4:
        plays_list = [plays_array[i] for i in range(plays_array.shape[0])]
    elif isinstance(plays, list):
        plays_list = plays
    else:
        plays_list = [plays_array]

    T = len(plays_list[0])
    n = plays_list[0].shape[1]
    num_runs = len(plays_list)

    fig, axes = plt.subplots(n, n, figsize=(4*n, 4*n))
    plt.subplots_adjust(hspace=0.4, wspace=0.4)

    if n == 1:
        axes = np.array([axes])
    axes_flat = axes.flatten()

    time_steps = np.arange(1, T + 1)
    # Always use blue color (same hue as single run)
    colors = ['#1f77b4'] * num_runs  # Blue color for all runs

    for i in range(n):
        for j in range(n):
            idx = i * n + j
            ax = axes_flat[idx]

            for run_idx, play in enumerate(plays_list):
                prob_over_time = play[:, i, j]
                ax.plot(
                    time_steps,
                    prob_over_time,
                    linewidth=2,
                    color=colors[run_idx],
                    alpha=0.7,
                    zorder=3,
                )

            ax.set_xlabel('Time Step', fontsize=10)
            ax.set_ylabel('Probability', fontsize=10)
            ax.set_title(
                rf'$(A_{{{i + 1}}}, A_{{{j + 1}}})$',
                fontsize=12,
                fontweight='bold',
            )
            ax.set_ylim(0, 1)
            ax.set_axisbelow(True)
            ax.grid(True, alpha=0.3, zorder=0)

    title = f' Average Empirical Play Over Time'
    if num_runs > 1:
        title += f' ({num_runs} runs)'
    if title_suffix:
        title += f' {title_suffix}'
    fig.suptitle(title, fontsize=16, fontweight='bold', y=0.92)
    plt.tight_layout(rect=[0, 0, 1, 0.92])

    if save_file:
        os.makedirs("results", exist_ok=True)
        fig.savefig(os.path.join("results", "plays_static.png"), dpi=150)
    plt.close(fig)


def visualize_strategies_static(strategies_list, title_prefix="", title_suffix=""):
    """Plots each player's raw strategy over time."""
    if len(strategies_list) == 0:
        return

    first_elem = strategies_list[0]

    if isinstance(first_elem, np.ndarray):
        strategies_list = [strategies_list]

    num_players = len(strategies_list[0])
    if num_players == 0:
        return

    T = len(strategies_list[0][0])
    n = strategies_list[0][0].shape[1]
    num_runs = len(strategies_list)

    fig, axes = plt.subplots(num_players, n, figsize=(4*n, 4*num_players))
    plt.subplots_adjust(hspace=0.4, wspace=0.4)

    if num_players == 1:
        axes = axes.reshape(1, -1)
    if n == 1:
        axes = axes.reshape(-1, 1)

    time_steps = np.arange(1, T + 1)
    colors = plt.cm.tab10(np.linspace(0, 1, num_runs)) if num_runs <= 10 else plt.cm.viridis(np.linspace(0, 1, num_runs))

    for player_idx in range(num_players):
        for action_idx in range(n):
            ax = axes[player_idx, action_idx]

            for run_idx, strategy_history in enumerate(strategies_list):
                prob_over_time = strategy_history[player_idx][:, action_idx]
                ax.plot(
                    time_steps,
                    prob_over_time,
                    linewidth=2,
                    color=colors[run_idx],
                    alpha=0.7,
                    zorder=3,
                )

            ax.set_xlabel('Time Step', fontsize=10)
            ax.set_ylabel('Probability', fontsize=10)
            ax.set_title(
                rf'Player {player_idx + 1}, $A_{{{action_idx + 1}}}$',
                fontsize=12,
                fontweight='bold',
            )
            ax.set_ylim(0, 1)
            ax.set_axisbelow(True)
            ax.grid(True, alpha=0.3, zorder=0)

    title = f'Average Player Strategies Over Time'
    if num_runs > 1:
        title += f' ({num_runs} runs)'
    if title_suffix:
        title += f' {title_suffix}'
    fig.suptitle(title, fontsize=16, fontweight='bold', y=0.92)
    plt.tight_layout(rect=[0, 0, 1, 0.90])

    os.makedirs("results", exist_ok=True)
    fig.savefig(os.path.join("results", "strategies_static.png"), dpi=150)
    plt.close(fig)


def run_multiple_learning(N, n, T, learner_types, jinx=False, random_steps=0, visualize=True, save_plays_static=True):
    """Performs learning on N number of n×n games, then plots the empirical play visualization."""
    all_avg_plays = []

    for i in tqdm(range(N), desc=f"Running {N} games"):
        if jinx:
            payoffs = get_jinx_game(n).payoffs
        else:
            payoffs = np.random.random((2, n, n))

        learning_alg = learning(payoffs, T, learner_types, random_steps)

        avg_plays = learning_alg.average_plays()
        all_avg_plays.append(avg_plays)

    if visualize:
        eta = None
        try:
            base_type = learner_types[0] if isinstance(learner_types, list) else learner_types
            if issubclass(base_type, MWULearner):
                eta = getattr(learning_alg.learners[0], "eta", None)
        except Exception:
            eta = None

        title_suffix = ""
        if eta is not None:
            title_suffix = rf"(MWU, random steps={random_steps})"

        visualize_plays_static(all_avg_plays, title_prefix="Average play", title_suffix=title_suffix, save_file=save_plays_static)

    return all_avg_plays


def generate_side_by_side_from_run_multiple(
    rs=(0, 1, 10, 100, 1000),
    N=5,
    n=2,
    T=10_000,
    learner_types=MWULearner,
    jinx=True,
    base_name=None,
):
    """Generate side-by-side comparison plots for different random_steps values."""
    os.makedirs("results", exist_ok=True)

    base_type = learner_types[0] if isinstance(learner_types, list) else learner_types
    if base_type.__name__.startswith("RM"):
        learner_label = "RM"
    elif base_type.__name__.startswith("MWU"):
        learner_label = "MWU"
    else:
        learner_label = base_type.__name__

    if base_name is None:
        base_name = f"{learner_label.lower()}_n={n}_N={N}_T={T}"

    plays_imgs = []

    for r in rs:
        print(f"\n{'='*60}")
        print(f"Experimenting with r = {r}")
        print(f"{'='*60}")

        # Run learning
        all_avg_plays = run_multiple_learning(
            N=N,
            n=n,
            T=T,
            learner_types=learner_types,
            jinx=jinx,
            random_steps=r,
            save_plays_static=False,
        )

        # Generate plays visualization temporarily to read the image
        visualize_plays_static(all_avg_plays, save_file=True)
        temp_plays_path = os.path.join("results", "plays_static.png")

        if not os.path.exists(temp_plays_path):
            raise FileNotFoundError(
                f"Expected {temp_plays_path} to exist; "
                "did visualize_* filenames change?"
            )

        plays_imgs.append(mpimg.imread(temp_plays_path))
        os.remove(temp_plays_path)

    num_r = len(rs)

    # Plays figure
    fig_p, axes_p = plt.subplots(1, num_r, figsize=(5 * num_r, 7.5))
    if num_r == 1:
        axes_p = [axes_p]

    for ax, img, r in zip(axes_p, plays_imgs, rs):
        ax.imshow(img)
        ax.set_axis_off()
        ax.set_title(f"r = {r}", fontsize=12)

    fig_p.suptitle(
        f"{learner_label}: Average Empirical Play Over Time\n"
        f"#Simulations = {N}, #Rounds = {T}",
        fontsize=14,
        y=0.85,
    )
    fig_p.tight_layout(rect=[0, 0, 1, 0.90])

    plays_out = os.path.join("results", f"{base_name}_plays_multi_r.png")
    fig_p.savefig(plays_out, dpi=600)
    plt.close(fig_p)

    print(f"Saved plays grid to: {plays_out}")

    return plays_out


## Set Hyperparameters

Modify these values to change the experiment configuration:


In [302]:
# Game size
n = 4

# Number of rounds
T = 1_000

# Random steps values to compare (r values)
rs = (0, 1, 10, 100)

# Learner type: RMLearner or MWULearner
learner_types = MWULearner

# Number of simulations to run for each r value
N = 100


## Run Experiment and Generate Plots


In [303]:
plays_path = generate_side_by_side_from_run_multiple(
    rs=rs,
    N=N,
    n=n,
    T=T,
    learner_types=learner_types
)

print(f"\n✅ Experiment complete!")
print(f"📈 Plays plot: {plays_path}")



Experimenting with r = 0


Running 100 games: 100%|██████████| 100/100 [00:01<00:00, 88.09it/s]



Experimenting with r = 1


Running 100 games: 100%|██████████| 100/100 [00:01<00:00, 84.53it/s]



Experimenting with r = 10


Running 100 games: 100%|██████████| 100/100 [00:01<00:00, 81.49it/s]



Experimenting with r = 100


Running 100 games: 100%|██████████| 100/100 [00:01<00:00, 88.53it/s]


Saved plays grid to: results/mwu_n=4_N=100_T=1000_plays_multi_r.png

✅ Experiment complete!
📈 Plays plot: results/mwu_n=4_N=100_T=1000_plays_multi_r.png


## Display Results

If you want to display the saved plots, run the cell below. Otherwise, you can download them from the `results/` folder in Colab's file browser.


In [304]:
from IPython.display import Image, display

# Display the generated plot
display(Image(plays_path))
